# Interactive per-rat session viewer (R4/R5/R6)

All three rats' sessions are loaded upfront so you can switch between them
without reloading. Widget at the bottom has a rat selector, session dropdown,
and Prev/Next buttons.

**Left column, top**: arena location of *attempts* (rat's COM at rear-start
frame). Split four ways by (matched vs missed) × (inside vs outside port zone),
same color scheme as the corresponding plot in `single_session.ipynb`.

**Left column, bottom**: session totals — rears, attempts, matches.

**Right column, top row**: matches split by port zone. PC1 and PC2 mean ± std
of matches whose COM at match-end is inside vs outside the port-exclusion
circle. Template band shaded behind.

**Right column, bottom row**: matched vs unmatched rears at their max-alignment
window inside `[rear_start, rear_end + 0.75 s]`.

Loading all three rats takes ~30 min the first time (~30 s/session on SMB).


In [41]:
# ---- config ----
RATS = ['R4', 'R5', 'R6']
n_components = 2
FPS = 20
W = 30              # template length in frames
DATA_ROOT = '/home/yutaka-sprague/olveczky_lab/Lab/CLIRB/data'
ALIGN_BUFFER_S    = 0.75
ACCEPTED_ERRORS   = 3
LICK_BOUT_GAP_S   = 2.0
ATTEMPT_WINDOW_S  = 10.0
DATE_START = '2026_06_05'  # inclusive
DATE_END   = '2026_12_31'


In [42]:
import os, sys, glob, time, warnings
import numpy as np
import pandas as pd
import scipy.io as _sio
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

sys.path.insert(0, '.')
from skeleton import normalize_skeleton_batch, project_to_pcs

warnings.filterwarnings('ignore', category=RuntimeWarning)
plt.rcParams['figure.dpi'] = 100
%matplotlib inline

import session_cache as sc


def _causal_median_5_kp(triang):
    pad = 4
    padded = np.concatenate([np.repeat(triang[:1], pad, axis=0), triang], axis=0)
    T = triang.shape[0]
    widx = np.arange(T)[:, None] + np.arange(5)[None, :]
    return np.median(padded[widx], axis=1).astype(np.float32)


def _project_pcs(triang, template):
    flipped = triang.astype(np.float32).copy()
    flipped[:, :, 2] = -flipped[:, :, 2]
    rotated, _, _ = normalize_skeleton_batch(flipped)
    rotated_med = _causal_median_5_kp(rotated)
    flat = rotated_med.reshape(rotated_med.shape[0], -1)
    centered = flat - template['feature_means']
    return centered @ template['pc_weights'].T


def _replay_matches(pcs_used, pc_template, bounds_arr, ae=ACCEPTED_ERRORS):
    T, _ = pcs_used.shape
    L = pc_template.shape[0]
    if T < L: return np.array([], dtype=int)
    base = np.arange(T - L + 1)[:, None]
    win = pcs_used[base + np.arange(L)]
    above = win >= (pc_template + bounds_arr)
    below = win <= (pc_template - bounds_arr)
    outside = (above | below).reshape(win.shape[0], -1).sum(axis=1)
    passes = outside <= ae
    matches, last = [], -L
    for k in range(len(passes)):
        if passes[k]:
            t = k + L - 1
            if t - last >= L:
                matches.append(t); last = t
    return np.array(matches, dtype=int)


def _detect_rears(pc0, thresh_rise=200.0, thresh_reset=150.0,
                   min_frames_above=int(0.5 * FPS)):
    _rears = []
    state, start, n_above = 'down', None, 0
    for i in range(len(pc0)):
        v = pc0[i]
        if state == 'down':
            if v > thresh_rise:
                state, start, n_above = 'up', i, 1
        else:
            if v > thresh_rise: n_above += 1
            if v < thresh_reset:
                _rears.append((start, i, n_above))
                state, start, n_above = 'down', None, 0
    if state == 'up':
        _rears.append((start, len(pc0) - 1, n_above))
    return [(s, e, n) for s, e, n in _rears if n > min_frames_above]


def _best_align_end(rs, re, pcs, template, buf):
    L = template.shape[0]
    e_min = max(rs, L - 1)
    e_max = min(re + buf, len(pcs) - 1)
    if e_max < e_min:
        return -1, np.nan
    ends = np.arange(e_min, e_max + 1)
    starts = ends - (L - 1)
    idx = starts[:, None] + np.arange(L)[None, :]
    win = pcs[idx, :template.shape[1]]
    mse = ((win - template[None]) ** 2).mean(axis=(1, 2))
    k = int(np.argmin(mse))
    return int(ends[k]), float(mse[k])


def _lick_frames_from_aligned(rat, session, n_frames):
    '''Decode lick onset frames from the cached lick_idx_sleap_cams vector
    (nearest-lick index; increments where a new lick becomes closest). The raw
    vector comes from processed/aligned_data.mat via the local session cache.'''
    _c = sc.get_raw_session(rat, session)
    if _c is None:
        return np.array([], dtype=int)
    lidx = np.asarray(_c['lick_idx_sleap_cams'], dtype=int).ravel()[:n_frames]
    if not len(lidx):
        return np.array([], dtype=int)
    d = np.diff(lidx, prepend=lidx[:1])
    return np.where(d > 0)[0].astype(int)


def _lick_bouts(lick_frames, gap_fr):
    if not len(lick_frames):
        return lick_frames
    gaps = np.diff(lick_frames, prepend=lick_frames[:1] - gap_fr - 1)
    return lick_frames[gaps >= gap_fr]


def _attempts(rear_end, lick_bout_frames, att_win_fr):
    n_rears = len(rear_end)
    a_ri, a_lf = [], []
    used = np.zeros(n_rears, dtype=bool)
    for lf in lick_bout_frames:
        cand = ((rear_end >= lf - att_win_fr) & (rear_end <= lf) & ~used)
        if cand.any():
            idx = np.where(cand)[0]
            j = idx[int(np.argmax(rear_end[idx]))]
            used[j] = True
            a_ri.append(j)
            a_lf.append(int(lf))
    return np.asarray(a_ri, dtype=int), np.asarray(a_lf, dtype=int)


In [43]:
def _find_template_file(sleap_dir, rat):
    cand = sorted(glob.glob(os.path.join(sleap_dir, f'{rat}_template_*.npz')))
    return cand[0] if cand else None


def _find_port(sleap_dir):
    m = sorted(glob.glob(os.path.join(sleap_dir, 'calibration', '*', 'water_port_location.npy')))
    return m[0] if m else None


def _min_port_from_config(sleap_dir):
    p = os.path.join(sleap_dir, 'campy_config.yaml')
    if not os.path.isfile(p): return 200.0
    with open(p) as f:
        for line in f:
            if 'min_port_distance' in line and ':' in line:
                try:
                    return float(line.split(':', 1)[1].split('#')[0].strip())
                except Exception:
                    pass
    return 200.0


def _train_state_from_config(sleap_dir):
    """behavior_params.train_state from campy_config.yaml. NaN if missing."""
    import yaml
    p = os.path.join(sleap_dir, 'campy_config.yaml')
    if not os.path.isfile(p): return np.nan
    try:
        with open(p) as f:
            cfg = yaml.safe_load(f)
        ts = cfg.get('behavior_params', {}).get('train_state', None)
        return int(ts) if ts is not None else np.nan
    except Exception:
        return np.nan


TRAIN_STATE_LABEL = {0: 'no tone', 1: 'tone only', 2: 'tone + water spurting'}


def _load_session(rat, session):
    sd = os.path.join(DATA_ROOT, rat, session, 'sleap')
    triang_path = os.path.join(sd, 'triang_keys_3D.npy')
    fm_path = os.path.join(sd, 'frame_mapping.csv')
    if not (os.path.isfile(triang_path) and os.path.isfile(fm_path)):
        return None
    # Raw inputs via the local disk cache (session_cache.py): first run reads SMB
    # + writes the cache, later runs read the local .npz. Derivation is unchanged.
    _c = sc.get_raw_session(rat, session)
    if _c is None:
        return None
    triang = _c['triang_3d']
    tmpl_path = _find_template_file(sd, rat)
    if tmpl_path is None:
        tmpl_path = os.path.join(DATA_ROOT, rat, 'templates', f'{rat}_template_1.npz')
        if not os.path.isfile(tmpl_path):
            return None
    tmpl_npz = np.load(tmpl_path, allow_pickle=True)
    template = {k: tmpl_npz[k] for k in tmpl_npz.files}
    template_file = os.path.basename(tmpl_path)
    train_state = _c['train_state']

    pcs_full = _project_pcs(triang, template)
    pcs_to_use = np.asarray(template['pcs_to_use']).ravel().astype(int)[:n_components]
    pcs_used = pcs_full[:, pcs_to_use]
    pc_template = template['template'][:, :n_components]
    bounds_arr = template['bounds_to_use'][:, :n_components] \
        if 'bounds_to_use' in template else \
        np.tile(template['feature_stds'][pcs_to_use] * float(template['bounds']),
                 (W, 1))

    # Match ends + interp handling
    match_ends = _replay_matches(pcs_used, pc_template, bounds_arr)
    real_cam0 = set(np.asarray(_c['frame_mapping_cam0'], dtype=int).tolist())
    n_frames = len(triang)
    is_interp = np.array([f not in real_cam0 for f in range(n_frames)])

    def _snap(t, max_shift=3):
        if not is_interp[t]: return t
        for d in range(1, max_shift + 1):
            if t - d >= 0 and not is_interp[t - d]: return t - d
            if t + d < n_frames and not is_interp[t + d]: return t + d
        return t

    match_ends_eff = np.array([_snap(t) for t in match_ends]) if len(match_ends) else np.array([], dtype=int)
    missed = is_interp[match_ends_eff] if len(match_ends_eff) else np.zeros(0, dtype=bool)

    com_world = triang.mean(axis=1)
    _pxy = np.asarray(_c['port_xy']).ravel()
    port_xy = _pxy[:2] if _pxy.size and np.isfinite(_pxy[:2]).all() else None
    min_port = float(_c['min_port_distance'])

    if port_xy is not None and len(match_ends_eff):
        mx = com_world[match_ends_eff, 0]
        my = com_world[match_ends_eff, 1]
        port_dist_match = np.hypot(mx - port_xy[0], my - port_xy[1])
        too_close = port_dist_match < min_port
    else:
        port_dist_match = np.full(len(match_ends_eff), np.nan)
        too_close = np.zeros(len(match_ends_eff), dtype=bool)

    # Rears
    pc0 = pcs_used[:, 0]
    rears = _detect_rears(pc0)
    n_rears = len(rears)
    rear_start = np.array([r[0] for r in rears], dtype=int)
    rear_end   = np.array([r[1] for r in rears], dtype=int)
    align_buf_fr = int(round(ALIGN_BUFFER_S * FPS))
    rear_align_end = np.full(n_rears, -1, dtype=int)
    rear_align_mse = np.full(n_rears, np.nan)
    for k in range(n_rears):
        e, mse = _best_align_end(rear_start[k], rear_end[k], pcs_used, pc_template, align_buf_fr)
        rear_align_end[k] = e
        rear_align_mse[k] = mse

    # For each match, which rear it overlaps (if any). Also which rears are matched.
    rear_matched = np.zeros(n_rears, dtype=bool)
    # `rear_has_outside_match`: at least one match with too_close=False overlaps this rear.
    rear_has_outside_match = np.zeros(n_rears, dtype=bool)
    # match -> rear index (or -1 if none). Used to compute matches-without-attempts.
    match_rear_idx = np.full(len(match_ends), -1, dtype=int)
    # rear -> effective match-end frame of its FIRST overlapping match (or -1).
    # Used to gate the port distance at the frame the online rig evaluated.
    rear_match_end = np.full(n_rears, -1, dtype=int)
    if n_rears and len(match_ends):
        m_s = match_ends - (W - 1)
        for j in range(len(match_ends)):
            hits = np.where((rear_start <= match_ends[j]) & (rear_end >= m_s[j]))[0]
            if len(hits):
                match_rear_idx[j] = int(hits[0])
                rear_matched[hits] = True
                for _h in hits:
                    if rear_match_end[_h] < 0:
                        rear_match_end[_h] = int(match_ends_eff[j])
                if not too_close[j]:
                    rear_has_outside_match[hits] = True

    # Rears inside port zone -- gated at the frame the online rig evaluates the
    # port test: the MATCH-END (trigger) frame for matched rears, else the
    # rear-END frame (top of the reared posture) for unmatched rears. NOT the
    # rear-START frame: the animal moves toward the port while rearing, so
    # rear-start mislabels rears that start far but complete near the port.
    if n_rears and port_xy is not None:
        rear_gate_frame = np.where(rear_match_end >= 0, rear_match_end, rear_end)
        rear_gate_frame = np.clip(rear_gate_frame, 0, n_frames - 1)
        _g_com = com_world[rear_gate_frame]
        rear_port_dist = np.hypot(_g_com[:, 0] - port_xy[0],
                                    _g_com[:, 1] - port_xy[1])
        rear_inside = rear_port_dist < min_port
    else:
        rear_gate_frame = np.zeros(n_rears, dtype=int)
        rear_port_dist = np.full(n_rears, np.nan)
        rear_inside = np.zeros(n_rears, dtype=bool)

    # Licks + bouts + attempts
    lick_frames = _lick_frames_from_aligned(rat, session, n_frames)
    lick_bout_frames = _lick_bouts(lick_frames, int(round(LICK_BOUT_GAP_S * FPS)))
    a_ri, a_lf = _attempts(rear_end, lick_bout_frames,
                             int(round(ATTEMPT_WINDOW_S * FPS)))
    n_attempts = len(a_ri)
    attempt_matched = rear_matched[a_ri] if n_attempts else np.zeros(0, dtype=bool)
    attempt_inside = rear_inside[a_ri] if n_attempts else np.zeros(0, dtype=bool)

    # Matches whose rear was NOT the target of any attempt attribution.
    # An attempt claims a specific rear index; matches whose match_rear_idx
    # is not in the attempted-rear set are 'no attempt'.
    attempted_rears = set(a_ri.tolist()) if n_attempts else set()
    match_has_attempt = np.array(
        [(match_rear_idx[j] != -1) and (int(match_rear_idx[j]) in attempted_rears)
         for j in range(len(match_ends))], dtype=bool)
    match_no_attempt_ends = match_ends_eff[~match_has_attempt] \
        if len(match_ends_eff) else np.array([], dtype=int)

    # rewards = attempts whose rear contained at least one match with match_end
    # outside the port zone (this is what the online rule would actually fire).
    # Equivalent to: online-fireable matches attributed to an attempt.
    if n_attempts:
        attempt_reward = rear_has_outside_match[a_ri]
    else:
        attempt_reward = np.zeros(0, dtype=bool)
    n_rewards = int(attempt_reward.sum())
    # matches outside port
    n_matches_outside = int((~too_close).sum()) if len(too_close) else 0

    return dict(
        rat=rat, session=session,
        template_file=template_file, train_state=train_state,
        triang=triang, com_world=com_world,
        port_xy=port_xy, min_port_distance=min_port,
        pcs_used=pcs_used, pc_template=pc_template, bounds_arr=bounds_arr,
        match_ends=match_ends, match_ends_eff=match_ends_eff,
        missed=missed, too_close=too_close,
        n_matches=len(match_ends_eff),
        n_matches_outside=n_matches_outside,
        match_rear_idx=match_rear_idx, match_has_attempt=match_has_attempt,
        match_no_attempt_ends=match_no_attempt_ends,
        rear_start=rear_start, rear_end=rear_end,
        rear_align_end=rear_align_end, rear_align_mse=rear_align_mse,
        rear_matched=rear_matched, rear_inside=rear_inside,
        rear_gate_frame=(rear_gate_frame if n_rears else np.zeros(0, dtype=int)),
        n_rears=n_rears,
        lick_frames=lick_frames, lick_bout_frames=lick_bout_frames,
        attempt_rear_idx=a_ri, attempt_lick_frame=a_lf,
        attempt_matched=attempt_matched, attempt_inside=attempt_inside,
        attempt_reward=attempt_reward,
        rear_has_outside_match=rear_has_outside_match,
        n_attempts=n_attempts, n_rewards=n_rewards,
        n_frames=n_frames,
    )


def _discover_sessions(rat):
    out = []
    rd = os.path.join(DATA_ROOT, rat)
    if not os.path.isdir(rd): return out
    for s in sorted(os.listdir(rd)):
        if len(s) < 10 or not (DATE_START <= s[:10] <= DATE_END): continue
        if os.path.isdir(os.path.join(rd, s, 'sleap')):
            out.append(s)
    return out


In [44]:
ALL_RAT_SESSIONS = {}   # rat -> {session_name: session_dict}
ALL_RAT_SORTED   = {}   # rat -> sorted list of session names actually loaded

for rat in RATS:
    sessions = _discover_sessions(rat)
    print(f'{rat}: {len(sessions)} sessions discovered')
    ALL_RAT_SESSIONS[rat] = {}
    for i, s in enumerate(sessions):
        t0 = time.time()
        try:
            r = _load_session(rat, s)
        except Exception as e:
            print(f'  [{i+1}/{len(sessions)}] {s}: FAILED {type(e).__name__}: {e}')
            continue
        if r is None:
            print(f'  [{i+1}/{len(sessions)}] {s}: missing files, skipped')
            continue
        ALL_RAT_SESSIONS[rat][s] = r
        print(f'  [{i+1}/{len(sessions)}] {s}: {time.time()-t0:.1f}s  '
              f'matches={r["n_matches"]} rears={r["n_rears"]} '
              f'attempts={r["n_attempts"]}')
    ALL_RAT_SORTED[rat] = sorted(ALL_RAT_SESSIONS[rat].keys())
    print(f'{rat}: {len(ALL_RAT_SORTED[rat])} sessions ready\n')

print('all rats loaded')
for rat in RATS:
    print(f'  {rat}: {len(ALL_RAT_SORTED[rat])} sessions')


R4: 37 sessions discovered
  [1/37] 2026_06_05_1: 0.5s  matches=4 rears=80 attempts=27
  [2/37] 2026_06_08_1: 0.6s  matches=5 rears=94 attempts=15
  [3/37] 2026_06_09_1: 0.5s  matches=18 rears=76 attempts=23
  [4/37] 2026_06_11_1: 0.8s  matches=8 rears=43 attempts=11
  [5/37] 2026_06_12_1: 0.8s  matches=20 rears=115 attempts=19
  [6/37] 2026_06_15_1: 0.7s  matches=18 rears=81 attempts=29
  [7/37] 2026_06_16_1: 0.6s  matches=15 rears=89 attempts=24
  [8/37] 2026_06_17_1: 0.9s  matches=22 rears=67 attempts=29
  [9/37] 2026_06_18_1: 1.0s  matches=14 rears=76 attempts=47
  [10/37] 2026_06_18_2: 1.0s  matches=34 rears=101 attempts=84
  [11/37] 2026_06_19_1: 0.8s  matches=59 rears=193 attempts=0
  [12/37] 2026_06_20_1: 0.8s  matches=53 rears=139 attempts=88
  [13/37] 2026_06_22_1: 0.7s  matches=41 rears=123 attempts=59
  [14/37] 2026_06_22_2: 0.7s  matches=43 rears=133 attempts=53
  [15/37] 2026_06_23_1: 0.7s  matches=45 rears=128 attempts=54
  [16/37] 2026_06_23_2: 0.8s  matches=36 rears=81

In [45]:
def _match_pc_segments(r):
    if not r['n_matches']:
        return (np.zeros((0, W, 2)), np.zeros((0, W, 2)))
    ends = r['match_ends_eff']
    starts = ends - (W - 1)
    ok = starts >= 0
    ends = ends[ok]; starts = starts[ok]; too_close = r['too_close'][ok]
    if not len(ends):
        return (np.zeros((0, W, 2)), np.zeros((0, W, 2)))
    idx = starts[:, None] + np.arange(W)[None, :]
    segs = r['pcs_used'][idx, :2]
    return segs[too_close], segs[~too_close]


def _rear_pc_segments(r):
    ok = r['rear_align_end'] >= 0
    if not ok.any():
        return (np.zeros((0, W, 2)), np.zeros((0, W, 2)))
    ends = r['rear_align_end'][ok]
    matched = r['rear_matched'][ok]
    starts = ends - (W - 1)
    keep = starts >= 0
    ends = ends[keep]; starts = starts[keep]; matched = matched[keep]
    if not len(ends):
        return (np.zeros((0, W, 2)), np.zeros((0, W, 2)))
    idx = starts[:, None] + np.arange(W)[None, :]
    segs = r['pcs_used'][idx, :2]
    return segs[matched], segs[~matched]


def _draw_attempt_arena(ax, r):
    """Attempts (5 categories) + matches-without-attempts (blue open circles).
    Attempt position = COM at that attempt's rear_start. Match-without-attempt
    position = COM at match_end_eff."""
    ax.clear()
    com = r['com_world']
    ax.plot(com[:, 0], com[:, 1], color='0.85', lw=0.5, zorder=1)
    port_xy = r['port_xy']
    min_port = r['min_port_distance']
    if port_xy is not None:
        ax.add_patch(plt.Circle(tuple(port_xy), min_port, color='red',
                                 alpha=0.08, zorder=1.5))
        ax.add_patch(plt.Circle(tuple(port_xy), min_port, fill=False,
                                 edgecolor='red', ls='--', lw=1.2, zorder=2))
    # Matches with no attempt attribution.
    m_no_att = r['match_no_attempt_ends']
    if len(m_no_att):
        mx = com[m_no_att, 0]; my = com[m_no_att, 1]
        ax.scatter(mx, my, s=80, facecolor='none', edgecolor='#1f77b4',
                    linewidth=1.6, zorder=2.5,
                    label=f'match, no attempt ({len(m_no_att)})')
    # Attempts split 4 ways.
    if r['n_attempts']:
        ri = r['attempt_rear_idx']
        rs = r['rear_start'][ri]
        ax_x = com[rs, 0]; ax_y = com[rs, 1]
        matched = r['attempt_matched']
        inside  = r['attempt_inside']
        m_out = matched & ~inside
        m_in  = matched & inside
        x_out = ~matched & ~inside
        x_in  = ~matched & inside
        ax.scatter(ax_x[m_out], ax_y[m_out], s=90, c='#2ca02c',
                    edgecolor='k', zorder=3,
                    label=f'matched, outside ({int(m_out.sum())})')
        ax.scatter(ax_x[m_in], ax_y[m_in], s=90, c='#2ca02c',
                    edgecolor='darkred', linewidth=1.5, zorder=3,
                    label=f'matched, inside ({int(m_in.sum())})')
        ax.scatter(ax_x[x_out], ax_y[x_out], s=110, c='#d62728',
                    marker='X', edgecolor='k', zorder=4,
                    label=f'missed, outside ({int(x_out.sum())})')
        ax.scatter(ax_x[x_in], ax_y[x_in], s=110, c='#d62728',
                    marker='X', edgecolor='darkred', linewidth=1.5, zorder=4,
                    label=f'missed, inside ({int(x_in.sum())})')
    if port_xy is not None:
        ax.scatter(*port_xy, s=250, marker='*', c='gold', edgecolor='k',
                    zorder=7, label='water port')
    ax.set_xlabel('x (world)')
    ax.set_ylabel('y (world)')
    ax.set_aspect('equal', adjustable='datalim')
    ax.set_title(f'Attempts (n={r["n_attempts"]}) + matches w/o attempt (n={len(m_no_att)})')
    ax.legend(loc='best', fontsize=6)


def _draw_totals_bar(ax, r):
    ax.clear()
    counts = [r['n_rears'], r['n_attempts'], r['n_matches'],
              r['n_matches_outside'], r['n_rewards']]
    labels = ['rears', 'attempts', 'matches',
              'matches\n(outside port)', 'rewards']
    colors = ['#666', '#d62728', '#1f77b4', '#7fbfff', '#2ca02c']
    x = np.arange(len(counts))
    ax.bar(x, counts, color=colors, alpha=0.85)
    y_top = max(counts) if max(counts) else 1
    for xi, v in enumerate(counts):
        ax.text(xi, v + y_top * 0.02, str(v), ha='center', fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=7)
    ax.set_ylabel('count')
    ax.set_title('Session totals')
    ax.set_ylim(0, y_top * 1.15)
    ax.grid(axis='y', alpha=0.2)


def _draw_pc_mean_std(ax, segs_a, segs_b, name_a, color_a, name_b, color_b,
                       pc_template, bounds_arr, pidx, title):
    ax.clear()
    t = np.arange(W) / FPS
    center = pc_template[:, pidx]
    band = bounds_arr[:, pidx]
    ax.fill_between(t, center - band, center + band, color='0.85',
                     label='template band', zorder=1)
    ax.plot(t, center, 'k--', lw=1.5, label='template', zorder=2)
    for segs, name, color in [(segs_a, name_a, color_a),
                                (segs_b, name_b, color_b)]:
        if not len(segs):
            continue
        m = segs[:, :, pidx].mean(axis=0)
        s = segs[:, :, pidx].std(axis=0)
        ax.plot(t, m, lw=2, color=color, label=f'{name} (n={len(segs)})', zorder=3)
        ax.fill_between(t, m - s, m + s, color=color, alpha=0.2, zorder=2)
    ax.set_xlabel('time within window (s)')
    ax.set_ylabel(f'PC{pidx + 1}')
    ax.set_title(title)
    ax.legend(fontsize=7, loc='best')


BIN_MIN = 5  # 5-min bins for the % line plot


def _draw_pct_line(ax, r):
    """4 traces on 5-min bins:
      rears% inside/outside port  (dashed lines)
      attempts% inside/outside port (solid lines)
    Numerator: matched-in-bin. Denominator: total-in-bin.
    """
    ax.clear()
    n_frames = r['n_frames']
    bin_fr = int(round(BIN_MIN * 60 * FPS))
    n_bins = max(1, int(np.ceil(n_frames / bin_fr)))
    edges = np.arange(n_bins + 1) * bin_fr
    centers_min = (edges[:-1] + bin_fr / 2) / FPS / 60.0

    def _pct_by_bin(events_fr, event_flag_matched, event_flag_zone):
        """Return per-bin % of events with (matched & in_zone) / (in_zone)."""
        out = np.full(n_bins, np.nan)
        for k in range(n_bins):
            lo, hi = edges[k], edges[k+1]
            in_bin = (events_fr >= lo) & (events_fr < hi) & event_flag_zone
            n = int(in_bin.sum())
            if n:
                out[k] = 100.0 * (event_flag_matched & in_bin).sum() / n
        return out

    # rears
    rs   = r['rear_start']
    rm   = r['rear_matched']
    r_in = r['rear_inside']
    r_pct_in  = _pct_by_bin(rs, rm, r_in)
    r_pct_out = _pct_by_bin(rs, rm, ~r_in)

    # attempts
    if r['n_attempts']:
        alf = r['attempt_lick_frame']
        am  = r['attempt_matched']
        a_in = r['attempt_inside']
        a_pct_in  = _pct_by_bin(alf, am, a_in)
        a_pct_out = _pct_by_bin(alf, am, ~a_in)
    else:
        a_pct_in = a_pct_out = np.full(n_bins, np.nan)

    ax.plot(centers_min, r_pct_in,  marker='s', ls='--', lw=1.4, color='#d62728',
             label='rears matched %, inside')
    ax.plot(centers_min, r_pct_out, marker='s', ls='--', lw=1.4, color='#1f77b4',
             label='rears matched %, outside')
    ax.plot(centers_min, a_pct_in,  marker='o', ls='-',  lw=1.6, color='#d62728',
             label='attempts matched %, inside')
    ax.plot(centers_min, a_pct_out, marker='o', ls='-',  lw=1.6, color='#1f77b4',
             label='attempts matched %, outside')
    ax.set_xlabel('session time (min)')
    ax.set_ylabel('% matched')
    ax.set_ylim(-5, 115)
    ax.set_title(f'Match rate per {BIN_MIN}-min bin (rears vs attempts, by port zone)')
    ax.grid(alpha=0.25)
    ax.legend(fontsize=6, loc='lower right', ncol=2)


def _train_state_str(ts):
    if not np.isfinite(ts):
        return 'state=?'
    lbl = TRAIN_STATE_LABEL.get(int(ts), f'state {int(ts)}')
    return f'state {int(ts)}: {lbl}'


def _redraw(rat, session):
    r = ALL_RAT_SESSIONS[rat][session]
    sess_list = ALL_RAT_SORTED[rat]
    suptitle = (f'{rat}/{session}   '
                 f'(session {sess_list.index(session) + 1} of {len(sess_list)})  |  '
                 f'template={r["template_file"]}  |  {_train_state_str(r["train_state"])}')
    fig.suptitle(suptitle, fontsize=11)
    _draw_attempt_arena(ax_arena, r)
    _draw_totals_bar(ax_bar, r)
    m_inside, m_outside = _match_pc_segments(r)
    _draw_pc_mean_std(ax_tl, m_inside, m_outside,
                       'inside port', '#d62728', 'outside port', '#1f77b4',
                       r['pc_template'], r['bounds_arr'], 0,
                       'Matches — PC1')
    _draw_pc_mean_std(ax_tr, m_inside, m_outside,
                       'inside port', '#d62728', 'outside port', '#1f77b4',
                       r['pc_template'], r['bounds_arr'], 1,
                       'Matches — PC2')
    r_matched, r_unmatched = _rear_pc_segments(r)
    _draw_pc_mean_std(ax_bl, r_matched, r_unmatched,
                       'matched rear', '#2ca02c', 'unmatched rear', '#d62728',
                       r['pc_template'], r['bounds_arr'], 0,
                       'Rears (max-align) — PC1')
    _draw_pc_mean_std(ax_br, r_matched, r_unmatched,
                       'matched rear', '#2ca02c', 'unmatched rear', '#d62728',
                       r['pc_template'], r['bounds_arr'], 1,
                       'Rears (max-align) — PC2')
    _draw_pct_line(ax_line, r)
    fig.canvas.draw_idle()


In [47]:
assert any(ALL_RAT_SORTED.values()), 'no sessions loaded for any rat'

%matplotlib qt

# --- layout via subplot_mosaic ---
# Left col: arena (2 rows tall) + totals bar (short row).
# Right col: 2x2 PC panels + line plot spanning the 3rd row.
mosaic = [
    ['arena', 'tl',   'tr'],
    ['arena', 'bl',   'br'],
    ['bar',   'line', 'line'],
]
fig = plt.figure(figsize=(17, 10))
axd = fig.subplot_mosaic(
    mosaic,
    width_ratios=[1.4, 1, 1],
    height_ratios=[1, 1, 0.7],
    gridspec_kw=dict(wspace=0.3, hspace=0.55),
)
ax_arena = axd['arena']
ax_bar   = axd['bar']
ax_tl    = axd['tl']
ax_tr    = axd['tr']
ax_bl    = axd['bl']
ax_br    = axd['br']
ax_line  = axd['line']

# --- widgets ---
rats_available = [r for r in RATS if ALL_RAT_SORTED.get(r)]
rat_selector = widgets.ToggleButtons(
    options=rats_available,
    value=rats_available[0],
    description='rat:',
    layout=widgets.Layout(width='300px'),
)
session_dropdown = widgets.Dropdown(
    options=ALL_RAT_SORTED[rats_available[0]],
    value=ALL_RAT_SORTED[rats_available[0]][0],
    description='session:',
    layout=widgets.Layout(width='320px'),
)
prev_btn = widgets.Button(description='◀ Prev', layout=widgets.Layout(width='90px'))
next_btn = widgets.Button(description='Next ▶', layout=widgets.Layout(width='90px'))
counter  = widgets.Label(value='')


def _update_counter():
    sess_list = ALL_RAT_SORTED[rat_selector.value]
    i = sess_list.index(session_dropdown.value)
    counter.value = f'{i + 1} / {len(sess_list)}'
    prev_btn.disabled = (i == 0)
    next_btn.disabled = (i == len(sess_list) - 1)


def _on_rat_change(change):
    if change['name'] == 'value':
        new_list = ALL_RAT_SORTED[change['new']]
        session_dropdown.unobserve(_on_session_change, names='value')
        session_dropdown.options = new_list
        session_dropdown.value = new_list[0]
        session_dropdown.observe(_on_session_change, names='value')
        _update_counter()
        _redraw(rat_selector.value, session_dropdown.value)


def _on_session_change(change):
    if change['name'] == 'value':
        _update_counter()
        _redraw(rat_selector.value, change['new'])


def _on_prev(_):
    sess_list = ALL_RAT_SORTED[rat_selector.value]
    i = sess_list.index(session_dropdown.value)
    if i > 0:
        session_dropdown.value = sess_list[i - 1]


def _on_next(_):
    sess_list = ALL_RAT_SORTED[rat_selector.value]
    i = sess_list.index(session_dropdown.value)
    if i < len(sess_list) - 1:
        session_dropdown.value = sess_list[i + 1]


rat_selector.observe(_on_rat_change, names='value')
session_dropdown.observe(_on_session_change, names='value')
prev_btn.on_click(_on_prev)
next_btn.on_click(_on_next)

_update_counter()
_redraw(rat_selector.value, session_dropdown.value)

controls = widgets.VBox([
    rat_selector,
    widgets.HBox([prev_btn, session_dropdown, next_btn, counter]),
])
display(controls)
